# Data Acquisition

This notebook is the single ingestion point for the AquaSense AI MVP.
It stays close to a production data pipeline by separating direct sensor telemetry,
soft-sensor estimates, sparse lab-confirmed samples, consent limits, compliance labels,
prediction targets, alert seeds, and dashboard-ready snapshots.

## 1. Setup

Import the libraries and define project paths. Generated data is written into `data/`,
shared config into `packages/shared/config/`, and demo report inputs into `reports/`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
SIM = ROOT / "data" / "simulated"
PROCESSED = ROOT / "data" / "processed"
CONFIG = ROOT / "packages" / "shared" / "config"
REPORTS = ROOT / "reports"

for path in [RAW, SIM, PROCESSED, CONFIG, REPORTS]:
    path.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(42)
ROOT

PosixPath('/Users/dannyyy/Documents/AQUASENSE AI')

## 2. Consent Limits

These are demo trade-effluent consent limits, not universal legal thresholds.
Production systems should load site-specific limits from each facility's permit or consent.

In [2]:
consent_limits = {
    "facility_id": "demo-food-processing-plant",
    "facility_name": "Demo Food Processing Plant",
    "sample_interval_minutes": 5,
    "prediction_horizons_minutes": [15, 30, 60],
    "amber_margin_pct": 15,
    "limits": {
        "ph_min": 6.0,
        "ph_max": 10.0,
        "temperature_max_c": 43.0,
        "cod_max_mg_l": 1500,
        "bod_max_mg_l": 900,
        "tss_max_mg_l": 800,
        "ammonia_max_mg_l": 180,
    },
}

(CONFIG / "demo_limits.json").write_text(json.dumps(consent_limits, indent=2))
consent_limits

{'facility_id': 'demo-food-processing-plant',
 'facility_name': 'Demo Food Processing Plant',
 'sample_interval_minutes': 5,
 'prediction_horizons_minutes': [15, 30, 60],
 'amber_margin_pct': 15,
 'limits': {'ph_min': 6.0,
  'ph_max': 10.0,
  'temperature_max_c': 43.0,
  'cod_max_mg_l': 1500,
  'bod_max_mg_l': 900,
  'tss_max_mg_l': 800,
  'ammonia_max_mg_l': 180}}

## 3. Direct Sensor Telemetry

This is the realistic real-time stream. COD and BOD are intentionally not treated
as direct probe readings. They will be estimated later and corrected by sparse lab data.

In [3]:
start = pd.Timestamp("2026-05-01 00:00:00")
periods = 30 * 24 * 12
timestamps = pd.date_range(start, periods=periods, freq="5min")

sensors = pd.DataFrame({
    "timestamp": timestamps,
    "facility_id": consent_limits["facility_id"],
})

minute = sensors["timestamp"].dt.hour * 60 + sensors["timestamp"].dt.minute
daily_cycle = np.sin(2 * np.pi * minute / 1440)
production = ((sensors["timestamp"].dt.hour >= 7) & (sensors["timestamp"].dt.hour <= 19)).astype(float)

sensors["flow_rate_lps"] = 42 + 18 * production + 7 * daily_cycle + RNG.normal(0, 2.5, len(sensors))
sensors["temperature_c"] = 24 + 4 * production + 2 * daily_cycle + RNG.normal(0, 0.8, len(sensors))
sensors["ph"] = 7.4 + 0.25 * daily_cycle + RNG.normal(0, 0.08, len(sensors))
sensors["turbidity_ntu"] = 55 + 35 * production + 10 * daily_cycle + RNG.normal(0, 6, len(sensors))
sensors["conductivity_us_cm"] = 1250 + 260 * production + 80 * daily_cycle + RNG.normal(0, 45, len(sensors))
sensors["dissolved_oxygen_mg_l"] = 6.8 - 0.018 * sensors["turbidity_ntu"] + RNG.normal(0, 0.25, len(sensors))
sensors["orp_mv"] = 210 - 0.7 * sensors["turbidity_ntu"] + RNG.normal(0, 12, len(sensors))
sensors["ammonia_mg_l"] = 18 + 10 * production + RNG.normal(0, 2.5, len(sensors))
sensors["uv254_abs"] = 0.25 + 0.004 * sensors["turbidity_ntu"] + RNG.normal(0, 0.025, len(sensors))

numeric_cols = sensors.select_dtypes("number").columns
sensors[numeric_cols] = sensors[numeric_cols].clip(lower=0)
sensors["sensor_status"] = "ok"
sensors["event_type"] = "normal"

sensors.head()

,timestamp,facility_id,flow_rate_lps,temperature_c,ph,turbidity_ntu,conductivity_us_cm,dissolved_oxygen_mg_l,orp_mv,ammonia_mg_l,uv254_abs,sensor_status,event_type
0,2026-05-01 00:00:00,demo-food-processing-plant,42.761793,24.704246,7.583163,47.714152,1253.298771,6.175771,145.890513,19.227143,0.370378,ok,normal
1,2026-05-01 00:05:00,demo-food-processing-plant,39.552744,25.428955,7.329990,54.237621,1213.397632,5.926011,162.121332,16.743174,0.427683,ok,normal
2,2026-05-01 00:10:00,demo-food-processing-plant,44.181464,23.620196,7.298181,41.990992,1227.484933,5.562195,173.896624,19.850248,0.461505,ok,normal
3,2026-05-01 00:15:00,demo-food-processing-plant,44.809234,22.662621,7.551415,43.606992,1285.460222,5.479209,159.875724,14.211741,0.400441,ok,normal
4,2026-05-01 00:20:00,demo-food-processing-plant,37.732502,24.573767,7.527489,53.865864,1278.825794,5.366136,170.315560,16.585168,0.455381,ok,normal


## 4. Incident Injection

Inject known operational events into direct sensors. This gives downstream rules and models
realistic precursors before a breach occurs.

In [4]:
incidents = [
    ("organic_overload", "2026-05-05 08:30", 180),
    ("solids_washout", "2026-05-09 13:00", 90),
    ("acid_spill", "2026-05-12 10:15", 75),
    ("alkali_spill", "2026-05-16 15:20", 60),
    ("ammonia_spike", "2026-05-20 06:45", 120),
    ("thermal_discharge", "2026-05-24 17:00", 100),
    ("sensor_drift", "2026-05-27 09:00", 240),
]

for event, start_time, duration_min in incidents:
    mask = sensors["timestamp"].between(pd.Timestamp(start_time), pd.Timestamp(start_time) + pd.Timedelta(minutes=duration_min))
    ramp = np.linspace(0.15, 1.0, mask.sum())
    sensors.loc[mask, "event_type"] = event

    if event == "organic_overload":
        sensors.loc[mask, ["uv254_abs", "turbidity_ntu", "dissolved_oxygen_mg_l"]] += np.c_[0.42*ramp, 70*ramp, -1.1*ramp]
    elif event == "solids_washout":
        sensors.loc[mask, "turbidity_ntu"] += 160 * ramp
    elif event == "acid_spill":
        sensors.loc[mask, "ph"] -= 2.0 * ramp
    elif event == "alkali_spill":
        sensors.loc[mask, "ph"] += 3.0 * ramp
    elif event == "ammonia_spike":
        sensors.loc[mask, "ammonia_mg_l"] += 190 * ramp
    elif event == "thermal_discharge":
        sensors.loc[mask, "temperature_c"] += 22 * ramp
    elif event == "sensor_drift":
        sensors.loc[mask, "conductivity_us_cm"] += 420 * ramp

missing_mask = sensors["timestamp"].between("2026-05-22 02:00", "2026-05-22 02:45")
flatline_mask = sensors["timestamp"].between("2026-05-28 11:00", "2026-05-28 12:00")
sensors.loc[missing_mask, ["ph", "turbidity_ntu", "uv254_abs"]] = np.nan
sensors.loc[missing_mask, "sensor_status"] = "missing_burst"
sensors.loc[flatline_mask, "sensor_status"] = "flatline"
sensors.loc[flatline_mask, "ph"] = sensors.loc[flatline_mask, "ph"].iloc[0]

sensors["event_type"].value_counts()

event_type
normal               8460
sensor_drift           49
organic_overload       37
ammonia_spike          25
thermal_discharge      21
solids_washout         19
acid_spill             16
alkali_spill           13
Name: count, dtype: int64

## 5. Soft-Sensor Estimates

Estimate hard-to-measure water-quality indicators from direct telemetry.
In production, these formulas would be replaced by calibrated models trained against lab samples.

In [5]:
estimates = sensors[["timestamp", "facility_id", "event_type", "sensor_status"]].copy()
estimates["estimated_tss_mg_l"] = 4.8 * sensors["turbidity_ntu"] + 0.04 * sensors["flow_rate_lps"] + RNG.normal(0, 28, len(sensors))
estimates["estimated_cod_mg_l"] = (
    250
    + 1.1 * estimates["estimated_tss_mg_l"]
    + 350 * sensors["uv254_abs"]
    + 0.08 * sensors["conductivity_us_cm"]
    - 20 * sensors["dissolved_oxygen_mg_l"]
    + RNG.normal(0, 70, len(sensors))
)
estimates["estimated_bod_mg_l"] = 0.50 * estimates["estimated_cod_mg_l"] + 8 * sensors["uv254_abs"] + RNG.normal(0, 35, len(sensors))

for col in ["estimated_tss_mg_l", "estimated_cod_mg_l", "estimated_bod_mg_l"]:
    estimates[col] = estimates[col].clip(lower=0)

estimates["quality_value_source"] = "soft_sensor_estimate"
estimates.head()

,timestamp,facility_id,event_type,sensor_status,estimated_tss_mg_l,estimated_cod_mg_l,estimated_bod_mg_l,quality_value_source
0,2026-05-01 00:00:00,demo-food-processing-plant,normal,ok,232.293283,609.722347,313.058763,soft_sensor_estimate
1,2026-05-01 00:05:00,demo-food-processing-plant,normal,ok,222.816820,584.344866,260.141857,soft_sensor_estimate
2,2026-05-01 00:10:00,demo-food-processing-plant,normal,ok,250.955578,741.322653,419.128019,soft_sensor_estimate
3,2026-05-01 00:15:00,demo-food-processing-plant,normal,ok,206.203928,677.524333,401.191480,soft_sensor_estimate
4,2026-05-01 00:20:00,demo-food-processing-plant,normal,ok,249.421527,743.034718,322.438840,soft_sensor_estimate


## 6. Lab-Confirmed Samples

Simulate sparse lab samples every six hours, plus extra samples during incidents.
These are not continuous, but they provide ground truth for compliance confidence and future calibration.

In [6]:
lab_times = pd.date_range(start, periods=30 * 4, freq="6h")
incident_times = sensors.loc[sensors["event_type"] != "normal", "timestamp"].iloc[::12]
lab_times = pd.Index(sorted(set(lab_times).union(set(incident_times))))

lab = sensors.merge(estimates, on=["timestamp", "facility_id", "event_type", "sensor_status"])
lab = lab[lab["timestamp"].isin(lab_times)].copy()

bias = np.select(
    [lab["event_type"].eq("organic_overload"), lab["event_type"].eq("solids_washout")],
    [1.08, 1.04],
    default=1.0,
)
lab["lab_tss_mg_l"] = (lab["estimated_tss_mg_l"] * bias + RNG.normal(0, 18, len(lab))).clip(lower=0)
lab["lab_cod_mg_l"] = (lab["estimated_cod_mg_l"] * bias + RNG.normal(0, 55, len(lab))).clip(lower=0)
lab["lab_bod_mg_l"] = (lab["estimated_bod_mg_l"] * bias + RNG.normal(0, 35, len(lab))).clip(lower=0)
lab["lab_ammonia_mg_l"] = (lab["ammonia_mg_l"] + RNG.normal(0, 3, len(lab))).clip(lower=0)
lab["lab_result_available"] = True

lab = lab[["timestamp", "facility_id", "lab_tss_mg_l", "lab_cod_mg_l", "lab_bod_mg_l", "lab_ammonia_mg_l", "lab_result_available"]]
lab.head(), lab.shape

(              timestamp                 facility_id  lab_tss_mg_l  \
 0   2026-05-01 00:00:00  demo-food-processing-plant    243.178566   
 72  2026-05-01 06:00:00  demo-food-processing-plant    355.187346   
 144 2026-05-01 12:00:00  demo-food-processing-plant    403.250767   
 216 2026-05-01 18:00:00  demo-food-processing-plant    373.300449   
 288 2026-05-02 00:00:00  demo-food-processing-plant    197.188859   
 
      lab_cod_mg_l  lab_bod_mg_l  lab_ammonia_mg_l  lab_result_available  
 0      744.558218    326.357163         13.861750                  True  
 72     780.015685    500.151408         21.853438                  True  
 144    888.807729    449.272867         36.189762                  True  
 216   1015.518215    441.750512         26.569857                  True  
 288    639.104133    327.593990         20.731490                  True  ,
 (135, 7))

## 7. Operational Compliance Dataset

Merge direct sensors, soft-sensor estimates, and lab results into one operational view.
Lab values are used when available; otherwise the system uses soft-sensor estimates with source flags.

In [7]:
operational = sensors.merge(estimates.drop(columns=["event_type", "sensor_status"]), on=["timestamp", "facility_id"], how="left")
operational = operational.merge(lab, on=["timestamp", "facility_id"], how="left")
operational["lab_result_available"] = operational["lab_result_available"].fillna(False)

for metric in ["tss", "cod", "bod"]:
    operational[f"{metric}_mg_l_for_compliance"] = operational[f"lab_{metric}_mg_l"].fillna(operational[f"estimated_{metric}_mg_l"])
    operational[f"{metric}_value_source"] = np.where(operational[f"lab_{metric}_mg_l"].notna(), "lab_confirmed", "soft_sensor_estimate")

operational["ammonia_mg_l_for_compliance"] = operational["lab_ammonia_mg_l"].fillna(operational["ammonia_mg_l"])
operational["ammonia_value_source"] = np.where(operational["lab_ammonia_mg_l"].notna(), "lab_confirmed", "online_sensor")

operational[[
    "timestamp", "ph", "turbidity_ntu", "estimated_cod_mg_l", "lab_cod_mg_l",
    "cod_mg_l_for_compliance", "cod_value_source"
]].head(10)

,timestamp,ph,turbidity_ntu,estimated_cod_mg_l,lab_cod_mg_l,cod_mg_l_for_compliance,cod_value_source
0,2026-05-01 00:00:00,7.583163,47.714152,609.722347,744.558218,744.558218,lab_confirmed
1,2026-05-01 00:05:00,7.329990,54.237621,584.344866,NaN,584.344866,soft_sensor_estimate
2,2026-05-01 00:10:00,7.298181,41.990992,741.322653,NaN,741.322653,soft_sensor_estimate
3,2026-05-01 00:15:00,7.551415,43.606992,677.524333,NaN,677.524333,soft_sensor_estimate
4,2026-05-01 00:20:00,7.527489,53.865864,743.034718,NaN,743.034718,soft_sensor_estimate
5,2026-05-01 00:25:00,7.412629,55.318683,804.367907,NaN,804.367907,soft_sensor_estimate
6,2026-05-01 00:30:00,7.463138,54.425130,491.447182,NaN,491.447182,soft_sensor_estimate
7,2026-05-01 00:35:00,7.470556,42.241790,673.448454,NaN,673.448454,soft_sensor_estimate
8,2026-05-01 00:40:00,7.454349,66.050853,768.095092,NaN,768.095092,soft_sensor_estimate
9,2026-05-01 00:45:00,7.345054,48.333519,642.684772,NaN,642.684772,soft_sensor_estimate


## 8. Compliance Rules

Apply deterministic rules to the operational compliance fields.
This keeps the current status explainable and auditable.

In [8]:
L = consent_limits["limits"]

checks = pd.DataFrame({
    "pH below min": operational["ph"] < L["ph_min"],
    "pH above max": operational["ph"] > L["ph_max"],
    "temperature above max": operational["temperature_c"] > L["temperature_max_c"],
    "COD above max": operational["cod_mg_l_for_compliance"] > L["cod_max_mg_l"],
    "BOD above max": operational["bod_mg_l_for_compliance"] > L["bod_max_mg_l"],
    "TSS above max": operational["tss_mg_l_for_compliance"] > L["tss_max_mg_l"],
    "ammonia above max": operational["ammonia_mg_l_for_compliance"] > L["ammonia_max_mg_l"],
})

amber = (
    (operational["ph"] <= L["ph_min"] * 1.15) |
    (operational["ph"] >= L["ph_max"] * 0.85) |
    (operational["temperature_c"] >= L["temperature_max_c"] * 0.85) |
    (operational["cod_mg_l_for_compliance"] >= L["cod_max_mg_l"] * 0.85) |
    (operational["bod_mg_l_for_compliance"] >= L["bod_max_mg_l"] * 0.85) |
    (operational["tss_mg_l_for_compliance"] >= L["tss_max_mg_l"] * 0.85) |
    (operational["ammonia_mg_l_for_compliance"] >= L["ammonia_max_mg_l"] * 0.85)
)

operational["breach_now"] = checks.any(axis=1)
operational["compliance_status"] = np.select([operational["breach_now"], amber], ["RED", "AMBER"], default="GREEN")
operational["breached_parameters"] = checks.apply(lambda r: ", ".join(r.index[r]) if r.any() else "None", axis=1)
operational["current_breach_reason"] = operational["breached_parameters"].replace("None", "No current breach")

operational["compliance_status"].value_counts()

compliance_status
GREEN    8557
AMBER      50
RED        33
Name: count, dtype: int64

## 9. Risk Drivers and Actions

Find the parameter closest to its limit and map it to an operator-friendly recommendation.

In [9]:
ratios = pd.DataFrame({
    "pH low": L["ph_min"] / operational["ph"],
    "pH high": operational["ph"] / L["ph_max"],
    "temperature": operational["temperature_c"] / L["temperature_max_c"],
    "COD": operational["cod_mg_l_for_compliance"] / L["cod_max_mg_l"],
    "BOD": operational["bod_mg_l_for_compliance"] / L["bod_max_mg_l"],
    "TSS": operational["tss_mg_l_for_compliance"] / L["tss_max_mg_l"],
    "ammonia": operational["ammonia_mg_l_for_compliance"] / L["ammonia_max_mg_l"],
})

actions = {
    "pH low": "Check acid dosing or chemical spill; neutralise before discharge.",
    "pH high": "Check alkali cleaning stream; divert and neutralise if needed.",
    "temperature": "Check cooling process and pause thermal discharge if needed.",
    "COD": "Inspect organic-load process stream; consider diversion to holding tank.",
    "BOD": "Inspect biodegradable organic load and treatment aeration capacity.",
    "TSS": "Inspect filtration or settling process; reduce solids discharge.",
    "ammonia": "Inspect nitrogen-rich stream and trigger confirmatory sample.",
}

operational["main_risk_driver"] = ratios.idxmax(axis=1)
operational["margin_to_limit_pct"] = ((1 - ratios.max(axis=1)) * 100).round(1)
operational["recommended_action"] = operational["main_risk_driver"].map(actions)

operational[["timestamp", "main_risk_driver", "margin_to_limit_pct", "recommended_action"]].head()

,timestamp,main_risk_driver,margin_to_limit_pct,recommended_action
0,2026-05-01 00:00:00,pH low,20.9,Check acid dosing or chemical spill; neutralis...
1,2026-05-01 00:05:00,pH low,18.1,Check acid dosing or chemical spill; neutralis...
2,2026-05-01 00:10:00,pH low,17.8,Check acid dosing or chemical spill; neutralis...
3,2026-05-01 00:15:00,pH low,20.5,Check acid dosing or chemical spill; neutralis...
4,2026-05-01 00:20:00,pH low,20.3,Check acid dosing or chemical spill; neutralis...


## 10. Prediction Targets

Create 15, 30, and 60-minute breach targets.
Each target is true if a breach occurs at least once in that future window.

In [10]:
steps_per_window = {15: 3, 30: 6, 60: 12}

for minutes, steps in steps_per_window.items():
    target = (
        operational["breach_now"]
        .shift(-1)
        .rolling(window=steps, min_periods=1)
        .max()
        .shift(-(steps - 1))
        .fillna(False)
        .astype(bool)
    )
    operational[f"breach_next_{minutes}min"] = target

target_cols = [f"breach_next_{m}min" for m in steps_per_window]
operational[target_cols].mean().mul(100).round(2).rename("positive_rate_pct")

breach_next_15min    0.54
breach_next_30min    0.75
breach_next_60min    1.17
Name: positive_rate_pct, dtype: float64

## 11. Model-Ready Features

Add compact lag and rolling features. The model sees direct sensors plus source-aware
operational compliance values, which is closer to a production wastewater system.

In [11]:
feature_cols = [
    "ph", "temperature_c", "flow_rate_lps", "turbidity_ntu",
    "conductivity_us_cm", "dissolved_oxygen_mg_l", "orp_mv", "uv254_abs",
    "ammonia_mg_l_for_compliance", "tss_mg_l_for_compliance",
    "cod_mg_l_for_compliance", "bod_mg_l_for_compliance",
]

model_df = operational.copy()
for col in feature_cols:
    model_df[f"{col}_lag_5min"] = model_df[col].shift(1)
    model_df[f"{col}_rolling_30min_mean"] = model_df[col].rolling(6, min_periods=1).mean()

model_df = model_df.dropna(subset=feature_cols).reset_index(drop=True)
model_df.shape

(8630, 64)

## 12. Alerts and Dashboard Snapshot

Prepare small datasets the backend and dashboard can use immediately.
Source fields let the UI distinguish lab-confirmed values from soft-sensor estimates.

In [12]:
alerts = model_df.loc[
    model_df["compliance_status"].isin(["AMBER", "RED"]) |
    model_df[["breach_next_15min", "breach_next_30min", "breach_next_60min"]].any(axis=1),
    [
        "timestamp", "facility_id", "compliance_status", "main_risk_driver",
        "current_breach_reason", "recommended_action",
        "cod_value_source", "bod_value_source", "tss_value_source",
        "breach_next_15min", "breach_next_30min", "breach_next_60min",
    ],
].copy()

latest = model_df.iloc[-1][[
    "timestamp", "facility_id", "compliance_status", "main_risk_driver",
    "margin_to_limit_pct", "recommended_action", "cod_value_source",
    "bod_value_source", "tss_value_source", "breach_next_15min",
    "breach_next_30min", "breach_next_60min",
]].to_dict()

latest["timestamp"] = str(latest["timestamp"])
latest["risk_level"] = "HIGH" if latest["breach_next_30min"] else latest["compliance_status"]
alerts.head(), latest

(               timestamp                 facility_id compliance_status  \
 1254 2026-05-05 08:30:00  demo-food-processing-plant             AMBER   
 1259 2026-05-05 08:55:00  demo-food-processing-plant             AMBER   
 1264 2026-05-05 09:20:00  demo-food-processing-plant             AMBER   
 1270 2026-05-05 09:50:00  demo-food-processing-plant             AMBER   
 1272 2026-05-05 10:00:00  demo-food-processing-plant             AMBER   
 
      main_risk_driver current_breach_reason  \
 1254              COD     No current breach   
 1259              COD     No current breach   
 1264              COD     No current breach   
 1270              COD     No current breach   
 1272              COD     No current breach   
 
                                      recommended_action      cod_value_source  \
 1254  Inspect organic-load process stream; consider ...         lab_confirmed   
 1259  Inspect organic-load process stream; consider ...  soft_sensor_estimate   
 1264  Inspe

## 13. Data Dictionary

Save a compact dictionary that explains which fields are direct readings,
estimated values, lab results, and model targets.

In [13]:
dictionary = pd.DataFrame([
    ("direct_sensor_readings.csv", "Direct telemetry only: pH, temperature, flow, turbidity, conductivity, DO, ORP, ammonia, UV254."),
    ("soft_sensor_estimates.csv", "Estimated COD, BOD, and TSS derived from direct telemetry."),
    ("lab_results.csv", "Sparse lab-confirmed COD, BOD, TSS, and ammonia samples."),
    ("operational_compliance_dataset.csv", "Merged production-style dataset used by rules, alerts, reports, and dashboard."),
    ("compliance_training_dataset.csv", "Model-ready dataset with labels, source flags, lag features, and rolling features."),
    ("*_value_source", "Whether a compliance value came from lab_confirmed, soft_sensor_estimate, or online_sensor."),
    ("breach_next_15min", "True when a breach occurs within the next 15 minutes."),
    ("breach_next_30min", "True when a breach occurs within the next 30 minutes."),
    ("breach_next_60min", "True when a breach occurs within the next 60 minutes."),
], columns=["field_or_file", "description"])

dictionary

,field_or_file,description
0,direct_sensor_readings.csv,"Direct telemetry only: pH, temperature, flow, ..."
1,soft_sensor_estimates.csv,"Estimated COD, BOD, and TSS derived from direc..."
2,lab_results.csv,"Sparse lab-confirmed COD, BOD, TSS, and ammoni..."
3,operational_compliance_dataset.csv,"Merged production-style dataset used by rules,..."
4,compliance_training_dataset.csv,"Model-ready dataset with labels, source flags,..."
5,*_value_source,Whether a compliance value came from lab_confi...
6,breach_next_15min,True when a breach occurs within the next 15 m...
7,breach_next_30min,True when a breach occurs within the next 30 m...
8,breach_next_60min,True when a breach occurs within the next 60 m...


## 14. Write Pipeline Files

Persist every dataset required by the MVP pipeline.
The notebook is the ingestion source; downstream services should read these artifacts.

In [14]:
paths = {
    "direct sensor stream": RAW / "direct_sensor_readings.csv",
    "soft-sensor estimates": PROCESSED / "soft_sensor_estimates.csv",
    "legacy unified demo stream": SIM / "wastewater_sensor_readings.csv",
    "lab results": RAW / "lab_results.csv",
    "operational dataset": PROCESSED / "operational_compliance_dataset.csv",
    "model dataset": PROCESSED / "compliance_training_dataset.csv",
    "alerts": PROCESSED / "alerts_seed.csv",
    "latest dashboard": PROCESSED / "latest_dashboard_snapshot.json",
    "data dictionary": PROCESSED / "data_dictionary.csv",
    "consent limits": CONFIG / "demo_limits.json",
}

sensors.to_csv(paths["direct sensor stream"], index=False)
estimates.to_csv(paths["soft-sensor estimates"], index=False)
operational.to_csv(paths["legacy unified demo stream"], index=False)
lab.to_csv(paths["lab results"], index=False)
operational.to_csv(paths["operational dataset"], index=False)
model_df.to_csv(paths["model dataset"], index=False)
alerts.to_csv(paths["alerts"], index=False)
dictionary.to_csv(paths["data dictionary"], index=False)
paths["latest dashboard"].write_text(json.dumps(latest, indent=2, default=str))

pd.DataFrame({"artifact": paths.keys(), "path": paths.values()})

,artifact,path
0,direct sensor stream,/Users/dannyyy/Documents/AQUASENSE AI/data/raw...
1,soft-sensor estimates,/Users/dannyyy/Documents/AQUASENSE AI/data/pro...
2,legacy unified demo stream,/Users/dannyyy/Documents/AQUASENSE AI/data/sim...
3,lab results,/Users/dannyyy/Documents/AQUASENSE AI/data/raw...
4,operational dataset,/Users/dannyyy/Documents/AQUASENSE AI/data/pro...
5,model dataset,/Users/dannyyy/Documents/AQUASENSE AI/data/pro...
6,alerts,/Users/dannyyy/Documents/AQUASENSE AI/data/pro...
7,latest dashboard,/Users/dannyyy/Documents/AQUASENSE AI/data/pro...
8,data dictionary,/Users/dannyyy/Documents/AQUASENSE AI/data/pro...
9,consent limits,/Users/dannyyy/Documents/AQUASENSE AI/packages...


## 15. Final Quality Checks

Confirm row counts, lab coverage, source mix, target rates, and missing sensor gaps.

In [15]:
summary = {
    "sensor_rows": len(sensors),
    "lab_rows": len(lab),
    "model_rows": len(model_df),
    "lab_coverage_pct": round(model_df["lab_result_available"].mean() * 100, 2),
    "current_breach_pct": round(model_df["breach_now"].mean() * 100, 2),
    "alert_rows": len(alerts),
}

target_rates = model_df[["breach_next_15min", "breach_next_30min", "breach_next_60min"]].mean().mul(100).round(2)
source_mix = model_df[["cod_value_source", "bod_value_source", "tss_value_source"]].apply(lambda s: s.value_counts())
missing = sensors[["ph", "turbidity_ntu", "uv254_abs"]].isna().sum().loc[lambda s: s > 0]

summary, target_rates, source_mix, missing

({'sensor_rows': 8640,
  'lab_rows': 135,
  'model_rows': 8630,
  'lab_coverage_pct': np.float64(1.56),
  'current_breach_pct': np.float64(0.38),
  'alert_rows': 116},
 breach_next_15min    0.54
 breach_next_30min    0.75
 breach_next_60min    1.17
 dtype: float64,
                       cod_value_source  bod_value_source  tss_value_source
 soft_sensor_estimate              8495              8495              8495
 lab_confirmed                      135               135               135,
 ph               10
 turbidity_ntu    10
 uv254_abs        10
 dtype: int64)

## 16. Lab vs Estimate Validation

This is the key production-style validation step. The real-time model does not assume
COD, BOD, and TSS are directly measured every five minutes. Instead, it estimates them
from online signals, then compares those estimates against sparse lab-confirmed samples.

In [16]:
validation = operational.loc[
    operational["lab_result_available"],
    [
        "timestamp", "event_type",
        "estimated_cod_mg_l", "lab_cod_mg_l",
        "estimated_bod_mg_l", "lab_bod_mg_l",
        "estimated_tss_mg_l", "lab_tss_mg_l",
    ],
].copy()

validation.head()

,timestamp,event_type,estimated_cod_mg_l,lab_cod_mg_l,estimated_bod_mg_l,lab_bod_mg_l,estimated_tss_mg_l,lab_tss_mg_l
0,2026-05-01 00:00:00,normal,609.722347,744.558218,313.058763,326.357163,232.293283,243.178566
72,2026-05-01 06:00:00,normal,869.627772,780.015685,453.926906,500.151408,386.016202,355.187346
144,2026-05-01 12:00:00,normal,879.990576,888.807729,430.359642,449.272867,409.985885,403.250767
216,2026-05-01 18:00:00,normal,994.652627,1015.518215,489.475981,441.750512,387.632142,373.300449
288,2026-05-02 00:00:00,normal,605.533159,639.104133,325.875768,327.593990,198.004184,197.188859


The table below quantifies how close each soft-sensor estimate is to lab truth.
In production, these metrics would be monitored continuously and used to trigger recalibration.

In [17]:
def regression_metrics(actual, predicted):
    err = predicted - actual
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err ** 2))
    mape = np.mean(np.abs(err / actual.replace(0, np.nan))) * 100
    r2 = 1 - np.sum(err ** 2) / np.sum((actual - actual.mean()) ** 2)
    return pd.Series({"MAE": mae, "RMSE": rmse, "MAPE_%": mape, "R2": r2})

validation_metrics = pd.DataFrame({
    "COD": regression_metrics(validation["lab_cod_mg_l"], validation["estimated_cod_mg_l"]),
    "BOD": regression_metrics(validation["lab_bod_mg_l"], validation["estimated_bod_mg_l"]),
    "TSS": regression_metrics(validation["lab_tss_mg_l"], validation["estimated_tss_mg_l"]),
}).T.round(3)

validation_metrics

,MAE,RMSE,MAPE_%,R2
COD,46.017,57.173,5.555,0.909
BOD,29.209,36.749,7.356,0.883
TSS,15.700,20.429,4.445,0.971


The correlation table is useful for a quick presentation slide: high correlation means
the online estimates track lab-confirmed movement well, even if calibration still needs tuning.

In [18]:
correlation = pd.Series({
    "COD_lab_vs_estimate": validation["lab_cod_mg_l"].corr(validation["estimated_cod_mg_l"]),
    "BOD_lab_vs_estimate": validation["lab_bod_mg_l"].corr(validation["estimated_bod_mg_l"]),
    "TSS_lab_vs_estimate": validation["lab_tss_mg_l"].corr(validation["estimated_tss_mg_l"]),
}).round(3)

correlation

COD_lab_vs_estimate    0.955
BOD_lab_vs_estimate    0.943
TSS_lab_vs_estimate    0.987
dtype: float64

Persist the validation outputs so reports, README visuals, or the dashboard can reuse them.

In [19]:
validation_path = PROCESSED / "lab_vs_estimate_validation.csv"
metrics_path = PROCESSED / "lab_vs_estimate_metrics.csv"

validation.to_csv(validation_path, index=False)
validation_metrics.to_csv(metrics_path)

pd.DataFrame({
    "artifact": ["lab-vs-estimate validation rows", "lab-vs-estimate metrics"],
    "path": [validation_path, metrics_path],
})

,artifact,path
0,lab-vs-estimate validation rows,/Users/dannyyy/Documents/AQUASENSE AI/data/pro...
1,lab-vs-estimate metrics,/Users/dannyyy/Documents/AQUASENSE AI/data/pro...


## 17. Research Framing for the Pitch

This MVP follows a published industrial pattern: use fast online measurements as proxies,
estimate slower lab parameters with a calibrated soft sensor, and validate those estimates
against lab-confirmed samples.

Useful references:

- Li et al. developed a UV-Vis spectroscopy calibration approach for online COD estimation in rural sewage effluent and compared PLS, SVM, and neural-network methods: https://doi.org/10.1039/C9RA10732K
- A Water Research study on sensor fusion used UV/Vis spectroscopy plus turbidity to monitor COD, TSS, and oil/grease in wastewater: https://doi.org/10.1016/j.watres.2011.12.005
- Recent wastewater soft-sensor literature frames COD and BOD prediction as a practical alternative when direct real-time measurement is slow, sparse, or expensive: https://pubmed.ncbi.nlm.nih.gov/39629300/

Presentation phrasing:

> AquaSense AI does not pretend COD and BOD are simple real-time probe values. It estimates them from online sensors such as UV254, turbidity, conductivity, dissolved oxygen, pH, temperature, and flow, then continuously compares those estimates with lab-confirmed samples for calibration and trust.